![Python Banner](../../Image/05_06_Baysian_CausalML.png)



# 6.4 Bayesian Quasi-Experiments



A quasi-experiment identifies an effect by a design: two groups and a post period, a cutoff, an interruption date, an instrument, or a donor pool. The Bayesian step turns that design's coefficient into a posterior. This notebook is the detail chapter for the five classes in `pycausalml.bayesian_causalML`:

| Class | Design parameter |
|---|---|
| `BayesianDifferenceInDifferences` | ATT under parallel trends, the coefficient on treated $\times$ post |
| `BayesianRegressionDiscontinuity` | Jump at a cutoff in a local linear regression |
| `BayesianInterruptedTimeSeries` | Immediate level shift, with the slope change stored beside it |
| `BayesianInstrumentalVariable` | Structural coefficient in Bayesian two-stage least squares |
| `BayesianSyntheticControl` | Mean post-period gap after nonnegative donor weights are fit to the pre-period |

The layouts follow [CausalPy](https://causalpy.readthedocs.io/en/latest/). The posteriors are conjugate normal draws or a Bayesian bootstrap. They do not sample a custom PyMC likelihood. Notebooks 6.1–6.5 are that PyMC sequence. This notebook is the native module.



## Overview

Each class reports the same summary keys: `estimate` (the posterior mean of the point fit's target coefficient), `posterior_mean`, `ci_lower`, `ci_upper`, and `std`. Draws stay on `draws_`. A region of practical equivalence can be passed as `rope=(low, high)` where the constructor accepts it; the summary then adds the posterior probability that the effect sits inside that band.

### What this tutorial covers

- The regression or weighting formula for each design, and the posterior attached to it
- A simulation in which the design parameter is known, for all five classes
- Banks in business in the 6th and 8th Federal Reserve districts, as a difference-in-differences and as a single-series interruption
- Age-cell mortality at the minimum legal drinking age
- Settler mortality as an instrument for expropriation protection (Acemoglu, Johnson, and Robinson, 2001)
- California Proposition 99 and a synthetic donor pool (Abadie, Diamond, and Hainmueller, 2010)

The real series are the CausalPy tables in the [CausalPy data directory](https://github.com/pymc-labs/CausalPy/tree/main/causalpy/data), Apache License 2.0.



## The algorithms

Throughout, a **conjugate Gaussian posterior** means the following. The conditional mean is linear, $Y = X\beta + \varepsilon$, and the prior on the slopes is a ridge of precision $\lambda$ (`prior_precision`, default $10^{-4}$). The intercept column is almost unpenalized. The posterior of $\beta$ given the residual variance is normal:

$$
\beta \mid \sigma^2,\; Y \sim N\!\left(\hat\beta,\; \sigma^2 (X^\top X + \Lambda)^{-1}\right),
$$

where $\Lambda$ is diagonal with $\lambda$ on the penalized entries and $\hat\beta$ is the ridge estimate. $\sigma^2$ is plugged in as the residual variance. Credible intervals are quantiles of draws from that normal. Difference-in-differences, regression discontinuity, and interrupted time series use this posterior. The reported effect is one coordinate of $\beta$.

A **Bayesian bootstrap** draws Dirichlet weights on the rows and refits the whole procedure under those weights. Instrumental variables and synthetic control use it, so a first-stage or a donor-weight estimation step is inside every draw.

### 1. Difference-in-differences

For a binary group indicator $D_i$ and a binary post indicator $P_t$,

$$
Y_{it} = a + b D_i + c P_t + \tau D_i P_t + \varepsilon_{it}.
$$

Under parallel trends, $\tau$ is the average treatment effect on the treated: the change in the treated group's mean, minus the change in the control group's mean. `fit(y, treated, post)` builds the four columns and stores the posterior of $\tau$ (`effect_index` 3). The class does not add unit fixed effects or cluster the residuals. With many units and a large between-unit variance, the interval describes the residual variance of this saturated regression. Aggregating to the group-time means, or using a two-group panel such as the bank districts below, matches the regression the class actually fits.

### 2. Regression discontinuity

Let $R$ be the running variable, $c$ the cutoff, and $D = 1\{R \ge c\}$ when treatment is not supplied. Center $R_c = R - c$. The local linear model is

$$
Y = a + b R_c + \tau D + d R_c D + \varepsilon.
$$

$\tau$ is the jump at the cutoff. Rows with $|R_c| > h$ are dropped when `bandwidth` is set. The window must contain at least four rows. The posterior is the Gaussian posterior of $\tau$ inside that window. Units that can set $R$ exactly onto one side of $c$ violate the design; the posterior does not test that.

### 3. Interrupted time series

For a single series and an intervention time $t^\star$,

$$
Y_t = a + b\, t + \tau\, 1\{t \ge t^\star\} + d\, (t - t^\star)_+ + \varepsilon_t,
$$

where $(t - t^\star)_+ = \max(t - t^\star, 0)$. The reported effect is the immediate level shift $\tau$. The slope change $d$ is stored as `slope_change_`. A level shift is identified from the break in the mean after the pre-period trend is partialled out. A second shock that arrives at $t^\star$ is absorbed into $\tau$. That is why a control series, when one exists, belongs in the difference-in-differences specification rather than in a second copy of this regression.

### 4. Instrumental variables

The first stage and the second stage are

$$
\begin{aligned}
D &= \alpha_0 + Z^\top \alpha_1 + X^\top \alpha_2 + \varepsilon_D, \\
Y &= \beta_0 + \beta_1 \hat D + X^\top \beta_2 + \varepsilon_Y.
\end{aligned}
$$

`fit(y, treatment, instrument, X=None)` runs weighted least squares twice. The point estimate uses equal weights. Each posterior draw draws a Dirichlet weight vector and repeats both stages. $\beta_1$ is the structural effect, a local average treatment effect when the instrument is binary and the usual monotonicity condition holds. A weak first stage shows up as a wide posterior, because reweighting the rows moves $\hat D$.

`predict(treatment, X)` evaluates the second-stage equation at an intervened treatment level. It does not re-solve the first stage.

### 5. Synthetic control

One treated unit has a pre-period vector $Y^{\mathrm{pre}}$ and a post-period vector $Y^{\mathrm{post}}$. Donor matrices $X^{\mathrm{pre}}$ and $X^{\mathrm{post}}$ have one column per donor. Weights $w$ solve

$$
\min_{w \ge 0,\; \sum_j w_j = 1} \sum_{t \in \mathrm{pre}} \omega_t \left(Y^{\mathrm{pre}}_t - X^{\mathrm{pre}}_{t} w\right)^2.
$$

The point fit uses $\omega_t = 1$. The synthetic post-period path is $X^{\mathrm{post}} w$, and the reported effect is the mean gap

$$
\hat\tau = \frac{1}{T_{\mathrm{post}}} \sum_{t \in \mathrm{post}} \left(Y^{\mathrm{post}}_t - X^{\mathrm{post}}_{t} w\right).
$$

Posterior draws replace $\omega$ with a Dirichlet draw on the pre-period dates and re-solve the weights. The donor pool is part of the design. Changing it after seeing $\hat\tau$ changes the target.



## Setup

Estimators are imported from `pycausalml.bayesian_causalML`. Real datasets are loaded with `causalpy.load_data` when CausalPy is installed, and from the same CSV on GitHub otherwise.


In [ ]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "src" / "pycausalml").is_dir():
            return cand
    raise FileNotFoundError(
        "PyCausalML repository was not found. Run this notebook from the repository."
    )

root = _repo_root()
data_dir = root / "Data"
if not data_dir.is_dir():
    raise FileNotFoundError(
        "The Data directory was not found. It should sit next to pyproject.toml."
    )

try:
    import pycausalml  # noqa: F401
    print("PyCausalML ready.")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[plot]"],
    )
    print("PyCausalML installed.")
print("Data directory:", data_dir)


In [ ]:

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pycausalml.bayesian_causalML import (
    BayesianDifferenceInDifferences,
    BayesianInstrumentalVariable,
    BayesianInterruptedTimeSeries,
    BayesianRegressionDiscontinuity,
    BayesianSyntheticControl,
)
from pycausalml.data import make_iv_data

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
CAUSALPY_FILES = {
    "banks": "banks.csv",
    "drinking": "drinking.csv",
    "risk": "AJR2001.csv",
    "schoolReturns": "schoolingReturns.csv",
    "california_prop99": "california_prop99.csv",
}

def load_causalpy(name: str) -> pd.DataFrame:
    """Load a CausalPy example dataset.

    Uses causalpy.load_data when that package is installed. Otherwise reads
    the same CSV from the CausalPy GitHub data directory.
    """
    try:
        import causalpy as cp
        frame = cp.load_data(name)
        print(f"Loaded {name!r} with causalpy.load_data. Shape {frame.shape}.")
        return frame
    except ImportError:
        url = (
            "https://raw.githubusercontent.com/pymc-labs/CausalPy/main/causalpy/data/"
            + CAUSALPY_FILES[name]
        )
        frame = pd.read_csv(url)
        print(f"CausalPy is not installed. Loaded {name!r} from")
        print(url)
        print("Install the package with: pip install 'pycausalml[causalpy]'")
        print(f"Shape {frame.shape}.")
        return frame


def show_posterior(draws, estimate, title, truth=None, xlabel="Effect"):
    draws = np.asarray(draws, dtype=float).ravel()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(draws, bins=30, color="#4C72B0", alpha=0.9, density=True)
    ax.axvline(estimate, color="#8B1E3F", lw=2, label=f"estimate {estimate:.3f}")
    low, high = np.quantile(draws, [0.025, 0.975])
    ax.axvspan(low, high, color="#4C72B0", alpha=0.18, label="95% credible interval")
    if truth is not None:
        ax.axvline(truth, color="black", ls="--", lw=1.5, label=f"known effect {truth:.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()



## Getting started with simulated data

Each simulation writes the design parameter into the outcome and then asks the matching class to recover it. The random number generator is fixed so a second run reprints the same interval.



### Difference-in-differences

Half the units are treated. The post period is the second half of a two-period layout, woven through the sample. The interaction equals $1.5$ and the residual scale is $0.15$.


In [ ]:

rng = np.random.default_rng(0)
n = 240
treated = np.repeat([0.0, 1.0], n // 2)
post = np.tile([0.0, 1.0], n // 2)
y_did = 1.0 + 0.2 * treated + 0.3 * post + 1.5 * treated * post + rng.normal(scale=0.15, size=n)
did = BayesianDifferenceInDifferences(n_draws=400, random_state=0)
did.fit(y_did, treated, post)
print(pd.Series(did.summary()).round(3))
show_posterior(did.draws_, did.effect_, "Simulated difference-in-differences", truth=1.5)



### Regression discontinuity

The running variable is uniform on $[-1, 1]$. Treatment begins at $0$. The jump is $1.2$ and the slope on either side is $0.4$.


In [ ]:

running = rng.uniform(-1, 1, size=400)
jump = (running >= 0).astype(float)
y_rd = 0.4 * running + 1.2 * jump + rng.normal(scale=0.1, size=400)
rd = BayesianRegressionDiscontinuity(cutoff=0.0, n_draws=400, random_state=0)
rd.fit(running, y_rd)
print(pd.Series(rd.summary()).round(3))
show_posterior(rd.draws_, rd.effect_, "Simulated regression discontinuity", truth=1.2)

grid = np.linspace(-1, 1, 200)
# Local linear mean using the fitted coefficients: intercept, slope, jump, slope change.
coef = rd.coef_
center = grid - rd.cutoff
treated_grid = (grid >= rd.cutoff).astype(float)
mean = coef[0] + coef[1] * center + coef[2] * treated_grid + coef[3] * center * treated_grid
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(running, y_rd, s=10, alpha=0.35, c="#4C72B0")
ax.plot(grid, mean, color="#8B1E3F", lw=2)
ax.axvline(0.0, color="black", ls="--", lw=1)
ax.set_xlabel("Running variable")
ax.set_ylabel("Outcome")
ax.set_title("Jump at the cutoff")
fig.tight_layout()
plt.show()



### Interrupted time series

Eighty dates. The pre-period slope is $0.05$. At date 40 the level jumps by $2$ and the slope does not change.


In [ ]:

time = np.arange(80, dtype=float)
y_its = 0.05 * time + np.where(time >= 40, 2.0, 0.0) + rng.normal(scale=0.05, size=80)
its = BayesianInterruptedTimeSeries(n_draws=400, random_state=0)
its.fit(time, y_its, intervention_time=40)
print(pd.Series(its.summary()).round(3))
print("Slope change:", round(its.slope_change_, 3))
show_posterior(its.draws_, its.effect_, "Simulated interrupted time series", truth=2.0)



### Instrumental variables

`make_iv_data` builds a confounded treatment, a valid instrument, and covariates. The structural coefficient in that generator is $1.5$. The bootstrap uses 80 draws so the notebook stays short; a published interval can use more.


In [ ]:

iv_data = make_iv_data(n=800, random_state=1)
iv = BayesianInstrumentalVariable(n_draws=80, random_state=1)
iv.fit(iv_data["Y"], iv_data["W"], iv_data["Z"], iv_data["X"])
print(pd.Series(iv.summary()).round(3))
show_posterior(iv.draws_, iv.effect_, "Simulated instrumental variable", truth=1.5)



### Synthetic control

Three donors with weights $(0.2, 0.5, 0.3)$ reproduce the treated pre-period exactly. The post-period gap is a constant $1$.


In [ ]:

pre = rng.normal(size=(12, 3))
weights = np.array([0.2, 0.5, 0.3])
treated_pre = pre @ weights
post_donors = rng.normal(size=(6, 3))
treated_post = post_donors @ weights + 1.0
sc = BayesianSyntheticControl(n_draws=20, random_state=0)
sc.fit(pre, treated_pre, post_donors, treated_post)
print("Donor weights:", np.round(sc.weights_, 3), "sum", round(float(sc.weights_.sum()), 3))
print(pd.Series(sc.summary()).round(3))
show_posterior(sc.draws_, sc.effect_, "Simulated synthetic control", truth=1.0)



## Use case: Federal Reserve districts, 1929–1934

`banks` records banks in business in the 6th and 8th Federal Reserve districts (Richardson and Troost; Angrist and Pischke, *Mastering Metrics*). The 6th district changed policy. The 8th district is the comparison. The post period starts in 1931, matching an intervention placed at 1930.5 in the CausalPy notes. The table is aggregated to one median per district-year so each row is the series the design describes.

The difference-in-differences uses both districts. The interrupted time series uses the 6th district alone, on purpose: the Depression is a second shock at the same dates, and a single series folds that shock into the level shift. The contrast between the two fits is the reason to prefer the control district when it exists.


In [ ]:

banks = load_causalpy("banks")
yearly = (
    banks.filter(items=["bib6", "bib8", "year"])
    .rename(columns={"bib6": "sixth", "bib8": "eighth"})
    .groupby("year")
    .median()
    .reset_index()
    .sort_values("year")
)
long = yearly.melt(id_vars="year", var_name="district", value_name="banks_open")
long["treated"] = (long["district"] == "sixth").astype(float)
long["post"] = (long["year"] >= 1931).astype(float)
print(yearly.round(1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(yearly["year"], yearly["sixth"], marker="o", label="6th district", color="#8B1E3F")
ax.plot(yearly["year"], yearly["eighth"], marker="o", label="8th district", color="#4C72B0")
ax.axvline(1930.5, color="black", ls="--", lw=1, label="Intervention")
ax.set_ylabel("Banks in business")
ax.set_xlabel("Year")
ax.set_title("Two Federal Reserve districts")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


In [ ]:

banks_did = BayesianDifferenceInDifferences(n_draws=400, rope=(-5, 5), random_state=0)
banks_did.fit(long["banks_open"], long["treated"], long["post"])
print("Difference-in-differences, 6th minus 8th after 1930:")
print(pd.Series(banks_did.summary()).round(3))
show_posterior(
    banks_did.draws_,
    banks_did.effect_,
    "Banks in business: DiD",
    xlabel="Change in banks in business",
)

banks_its = BayesianInterruptedTimeSeries(n_draws=400, random_state=0)
banks_its.fit(yearly["year"].to_numpy(float), yearly["sixth"].to_numpy(float), intervention_time=1931)
print("Interrupted series, 6th district alone:")
print(pd.Series(banks_its.summary()).round(3))
print("Slope change:", round(banks_its.slope_change_, 3))
show_posterior(
    banks_its.draws_,
    banks_its.effect_,
    "Banks in business: ITS on the 6th district",
    xlabel="Level shift",
)



Read the DiD posterior as the interaction in a six-year, two-district regression. The ROPE of $\pm 5$ banks is a scale for "small" relative to a district that has on the order of a hundred banks in business. It is not a clinical or regulatory margin. The ITS level shift on the 6th district does not subtract the path of the 8th. If the two posteriors disagree, the design difference is the reason.



## Use case: mortality at age 21

`drinking` is the age-cell mortality series from the CausalPy regression-discontinuity example. The running variable is `agecell`, the midpoint of each age cell in years. The cutoff is the minimum legal drinking age, 21. The outcome is the all-cause death rate `all`. The bandwidth is one year, so the local linear fit uses cells in $[20, 22]$.


In [ ]:

drinking = load_causalpy("drinking")
print("Columns include agecell and all. Age runs from",
      round(float(drinking["agecell"].min()), 2), "to",
      round(float(drinking["agecell"].max()), 2))
rdd = BayesianRegressionDiscontinuity(cutoff=21, bandwidth=1.0, n_draws=400, random_state=0)
rdd.fit(drinking["agecell"].to_numpy(float), drinking["all"].to_numpy(float))
print("Rows inside the one-year window:", rdd.n_window_)
print(pd.Series(rdd.summary()).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(drinking["agecell"], drinking["all"], s=18, c="#4C72B0")
ax.axvline(21, color="#8B1E3F", ls="--", label="Age 21")
ax.set_xlabel("Age")
ax.set_ylabel("All-cause death rate")
ax.set_title("Death rate by age cell")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
show_posterior(rdd.draws_, rdd.effect_, "Jump in the death rate at age 21", xlabel="Jump at 21")



The jump is the vertical gap between the two local lines at 21, in death-rate units. Ages outside $[20, 22]$ do not enter the posterior. A bandwidth is a design choice: a wider window uses more cells and leans harder on the linear slope.



## Use case: colonial origins

`risk` is the Acemoglu, Johnson, and Robinson (2001) country table (`AJR2001.csv`). Log settler mortality `logmort0` is the instrument. Protection against expropriation `risk` is the endogenous treatment. Log GDP per capita `loggdp` is the outcome. The exclusion restriction — settler mortality affects present income only through institutions — is disputed. The posterior below is the two-stage least squares posterior under that restriction. It is not a test of the restriction.


In [ ]:

ajr = load_causalpy("risk").dropna(subset=["loggdp", "risk", "logmort0"])
print("Countries:", len(ajr))
ajr_iv = BayesianInstrumentalVariable(n_draws=200, random_state=0)
ajr_iv.fit(ajr["loggdp"].to_numpy(float), ajr["risk"].to_numpy(float), ajr["logmort0"].to_numpy(float))
print(pd.Series(ajr_iv.summary()).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(ajr["logmort0"], ajr["risk"], s=28, c="#4C72B0")
ax.set_xlabel("Log settler mortality")
ax.set_ylabel("Expropriation protection")
ax.set_title("First-stage cloud")
fig.tight_layout()
plt.show()
show_posterior(
    ajr_iv.draws_,
    ajr_iv.effect_,
    "Log GDP on expropriation protection",
    xlabel="IV coefficient",
)



## Use case: California Proposition 99

`california_prop99` is annual per-capita cigarette sales for 39 states, 1970–2000 (Abadie, Diamond, and Hainmueller, 2010). California raised the cigarette tax in 1989. The other states are the donor pool. Weights are nonnegative and sum to one. The point estimate matches the pre-1989 path with equal date weights. Bootstrap draws that match California less well on some pre-period years can move the mean gap, so the posterior mean need not equal the point estimate.


In [ ]:

prop = load_causalpy("california_prop99").set_index("Year").sort_index().astype(float)
treated_name = "California"
pre_years = prop.index < 1989
donors = prop.drop(columns=[treated_name])
synth = BayesianSyntheticControl(n_draws=25, random_state=0)
synth.fit(
    donors.loc[pre_years].to_numpy(),
    prop.loc[pre_years, treated_name].to_numpy(),
    donors.loc[~pre_years].to_numpy(),
    prop.loc[~pre_years, treated_name].to_numpy(),
)
weights = pd.Series(synth.weights_, index=donors.columns, name="weight")
print(weights.sort_values(ascending=False).head(8).round(3))
print("Average post-1988 gap, packs per capita:")
print(pd.Series(synth.summary()).round(2))

synth_pre = donors.loc[pre_years].to_numpy() @ synth.weights_
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(prop.index, prop[treated_name], label="California", color="#8B1E3F")
ax.plot(
    list(prop.index[pre_years]) + list(prop.index[~pre_years]),
    np.concatenate([synth_pre, synth.synth_post_]),
    label="Synthetic California",
    color="#4C72B0",
)
ax.axvline(1989, color="black", ls="--", lw=1, label="Proposition 99")
ax.set_ylabel("Cigarette packs per capita")
ax.set_title("California and its synthetic control")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
show_posterior(
    synth.draws_,
    synth.effect_,
    "Mean post-1988 gap",
    xlabel="Packs per capita",
)



### How to read a quasi-experimental posterior

- The interval is the uncertainty of the design coefficient under the likelihood the class uses. Parallel trends, no manipulation of the running variable, a isolated interruption, a valid instrument, and a fixed donor pool are outside that likelihood.
- Difference-in-differences here is the four-term regression. It does not cluster by state or county and it does not fit staggered adoption.
- A synthetic-control posterior recenters donor weights. It does not add a new donor.
- Pre-period fit, density at the cutoff, and the first-stage coefficient are the checks that decide whether the posterior is worth quoting.



## Summary and conclusion

The five classes share a summary and differ in the column they call the effect. Difference-in-differences, regression discontinuity, and interrupted time series draw that column from a ridge-normal posterior. Instrumental variables and synthetic control redraw the whole procedure under Dirichlet weights so the first stage, or the donor weights, moves with the data.

Simulations with a known jump, level shift, IV coefficient, interaction, and post-period gap recover those constants inside the plotted intervals. The bank panel shows the same years as a two-district interaction and as a single-series level shift. The drinking-age jump, the AJR two-stage coefficient, and the Proposition 99 gap are the same summaries on the CausalPy tables, each with the design assumption written next to the histogram.



## Scientific terminology

| Term | Meaning in this notebook |
|---|---|
| ATT | Average treatment effect on the treated. In the DiD regression it is $\tau$ on $D \times P$. |
| Parallel trends | Absent treatment, the treated and control means would have changed by the same amount. |
| Running variable | The score that assigns treatment when it crosses a cutoff. |
| Bandwidth | The window around the cutoff that enters the local linear fit. |
| Level shift | The immediate change in the mean at an intervention date, after the pre-period slope. |
| LATE | Local average treatment effect: the IV coefficient for units whose treatment responds to the instrument. |
| Bayesian bootstrap | Dirichlet weights on rows or on pre-period dates. Each weighted refit is one draw. |
| Donor pool | The untreated units allowed a positive weight in the synthetic control. |
| ROPE | A band of effects treated as practically small. The summary reports the posterior probability of that band. |
